In [ ]:
# app/modules/auth/test.ipynb

# Импорт зависимостей
import requests
import json

# Базовая конфигурация
base_url = "http://127.0.0.1:8000/auth"
headers = {"Content-Type": "application/json"}

In [ ]:
# POST /token — авторизация администратора
payload = {
    "username": "admin",
    "password": "admin"
}
resp = requests.post(f"{base_url}/token", data=payload)
print("Admin login status:", resp.status_code)
token_admin = resp.json().get("access_token")
headers_admin = {"Authorization": f"Bearer {token_admin}"}
print("Admin token:", token_admin)

In [ ]:
# POST /register — регистрация обычного пользователя
payload = {
    "login": "user2",
    "name": "User",
    "password": "12345"
}
resp = requests.post(f"{base_url}/register", headers=headers, json=payload)
print("User registration status:", resp.status_code)
if resp.status_code == 201:
    # Успешно
    user_id = resp.json().get("id")
    print("User ID:", user_id)
else:
    # Ошибка — выводим детали
    print("Error detail:", resp.json())

In [ ]:
# POST /token — авторизация обычного пользователя
payload = {
    "username": "user2",
    "password": "12345"
}
resp = requests.post(f"{base_url}/token", data=payload)
print("User login status:", resp.status_code)
token_user = resp.json().get("access_token")
headers_user = {"Authorization": f"Bearer {token_user}"}
print("User token:", token_user)

In [ ]:
# GET /users — список пользователей (только админ)
resp = requests.get(f"{base_url}/users", headers=headers_admin)
print("Get all users (admin) status:", resp.status_code)
if resp.status_code == 200:
    users = resp.json()
    for user in users:
        print(user)
else:
    print("Error detail:", resp.json())

In [ ]:
# GET /users — попытка получения списка пользователей обычным пользователем (должно быть 403)
resp = requests.get(f"{base_url}/users", headers=headers_user)
print("Get all users (user) status:", resp.status_code)
print(resp.json())

In [ ]:
# GET /users/{id} — получить пользователя по ID (админом)
resp = requests.get(f"{base_url}/users/{user_id}", headers=headers_admin)
print("Get user by ID (admin) status:", resp.status_code)
print(resp.json())

In [ ]:
# GET /users/{id} — получить свои данные по ID (обычным пользователем)
resp = requests.get(f"{base_url}/users/{user_id}", headers=headers_user)
print("Get self by ID (user) status:", resp.status_code)
print(resp.json())

In [ ]:
# GET /users/{id} — попытка получения чужих данных по ID (обычным пользователем)
resp = requests.get(f"{base_url}/users/1", headers=headers_user)  # чужой админ
print("Get admin by ID (user) status:", resp.status_code)
print(resp.json())

In [ ]:
# PUT /users/{id} — обновление своих данных обычным пользователем
payload = {"name": "User One Updated", "password": "67890"}
resp = requests.put(f"{base_url}/users/{user_id}", headers=headers_user, data=json.dumps(payload))
print("Update self (user) status:", resp.status_code)
print(resp.json())

In [ ]:
# DELETE /users/{id} — удаление аккаунта админом
resp = requests.delete(f"{base_url}/users/{user_id}", headers=headers_admin)
try:
    detail = resp.json().get("detail", resp.text)
except Exception:
    detail = resp.text
print(f"Status {resp.status_code}: {detail}")